# Exploratory Data Analysis (EDA) on Titanic Dataset

This beginner-friendly notebook explores passenger survival records with Pandas, NumPy, Matplotlib, and Seaborn. We will inspect the data, clean selected missing values, compare groups, and save six charts. The findings below are calculated from the included CSV; they are descriptive and do not imply that one factor caused an outcome.

Run the notebook from VS Code with the project virtual environment selected as its kernel. The source functions in `src/eda.py` are shared with the command-line version.

## 1. Import libraries and load the CSV

Pandas reads the CSV into a DataFrame, NumPy supports numeric calculations, and Seaborn/Matplotlib create charts. We add the project folder to Python's import path so this notebook can reuse the tested cleaning and plotting functions. The loader accepts CSV files from the project's data folder, checks their size, format, and required columns, and reports a short error for invalid files. See `SECURITY.md` for the validation details.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda import (
    clean_dataset,
    create_charts,
    create_group_summaries,
    create_key_findings,
    load_dataset,
)

DATA_PATH = PROJECT_ROOT / "data" / "titanic.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
raw_df = load_dataset(DATA_PATH)
print(f"Loaded {raw_df.shape[0]} rows and {raw_df.shape[1]} columns.")

## 2. Explore the dataset structure

`head()` and `tail()` preview records, `shape` and `columns` describe the table's dimensions and labels, and `info()` shows data types and non-null counts. `describe()` summarizes numeric columns. We also separate columns by their stored data type; `pclass` is numeric in the file but represents an ordered category.

In [ ]:
display(raw_df.head())
display(raw_df.tail())
print("Shape (rows, columns):", raw_df.shape)
print("Column names:", raw_df.columns.tolist())
raw_df.info()
display(raw_df.describe().T)

numeric_columns = raw_df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = raw_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)
print("pclass is stored as a number, but describes ordered passenger classes.")

## 3. Check missing values and duplicate rows

Before changing anything, count missing values in every column and count exact duplicate rows. We remove duplicates, then use the median for `Age` and `Fare` and the most common value (mode) for `Embarked`. These choices preserve all rows that have useful passenger information while using simple, explainable replacements.

In [ ]:
print("Missing values before cleaning:")
display(raw_df.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", int(raw_df.duplicated().sum()))

cleaned_df, duplicate_count = clean_dataset(raw_df)
print(f"Removed {duplicate_count} duplicate rows.")
print("Cleaned shape:", cleaned_df.shape)
print("Missing values after cleaning:")
display(cleaned_df.isna().sum().to_frame("remaining_missing"))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cleaned_path = OUTPUT_DIR / "titanic_cleaned.csv"
cleaned_df.to_csv(cleaned_path, index=False)
print(f"Saved cleaned dataset to {cleaned_path}")

## 4. Explore survival, age, fare, and embarkation

The survival column uses `1` for survived and `0` for did not survive, so its mean is the survival proportion. Group summaries make comparisons easier to read. Age and fare summaries describe the cleaned data; the embarkation table also includes the number of passengers in each group.

In [ ]:
survival_rate = cleaned_df["survived"].mean()
average_fare = cleaned_df["fare"].mean()
median_age = np.median(cleaned_df["age"].to_numpy())
print(f"Overall survival rate: {survival_rate:.1%}")
print(f"Average ticket fare: ${average_fare:.2f}")
print(f"Median age after cleaning: {median_age:.1f} years")

summaries = create_group_summaries(cleaned_df)
for title, summary in summaries.items():
    print("\n" + title.replace("_", " ").title())
    display(summary)

## 5. Create and save visualizations

The charts show outcome counts, age distribution, survival by gender and passenger class, numeric-feature correlations, and the spread of passenger ages. The reusable chart function labels axes and titles and saves high-resolution PNG files in `outputs/charts/`.

In [ ]:
charts_dir = OUTPUT_DIR / "charts"
chart_paths = create_charts(cleaned_df, charts_dir)
print(f"Created {len(chart_paths)} charts:")
for chart_path in chart_paths:
    print("-", chart_path.relative_to(PROJECT_ROOT))

first_chart = plt.imread(chart_paths[0])
plt.figure(figsize=(7, 5))
plt.imshow(first_chart)
plt.axis("off")
plt.title("Example saved chart: survival counts")
plt.show()

## 6. Findings and conclusion

The following findings are calculated from the cleaned dataset each time the notebook runs. They describe this dataset and should not be interpreted as causal claims. The closing note summarizes what the cleaning and analysis demonstrate.

In [ ]:
findings = create_key_findings(cleaned_df)
for number, finding in enumerate(findings, start=1):
    print(f"{number}. {finding}")

print(
    "\nConclusion: This exploration shows how to inspect and clean a real-world table "
    "before comparing outcomes. Missing Age values were filled with the median and "
    "missing Embarked values with its mode; Fare uses median imputation when needed, "
    "although this CSV has no missing fares. Exact duplicate rows were "
    "removed. The group summaries and charts make differences in observed survival "
    "by gender, passenger class, and embarkation port easier to examine. Pandas "
    "handled data loading, cleaning, and aggregation; NumPy supported numeric "
    "summaries; and Matplotlib/Seaborn made the patterns visible."
)